In [1]:
import pandas as pd
import numpy as np


In [2]:
customers = pd.read_csv("../data/olist_customers_dataset.csv")
orders = pd.read_csv("../data/olist_orders_dataset.csv")
payments = pd.read_csv("../data/olist_order_payments_dataset.csv")

print(customers.shape)
print(orders.shape)
print(payments.shape)


(99441, 5)
(99441, 8)
(103886, 5)


In [ ]:
# Customers only ID needed
customers = customers[['customer_id']]

# Orders = identifiers + timestamp
orders = orders[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']]

# Payment = monetary value
payments = payments[['order_id', 'payment_value']]


In [4]:
# Drop rows with missing critical values
orders = orders.dropna(subset=['order_id', 'customer_id', 'order_purchase_timestamp'])

# Convert timestamp safely
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp'],
    errors='coerce'
)

# Drop rows where timestamp conversion failed
orders = orders.dropna(subset=['order_purchase_timestamp'])

# Keep only delivered orders
orders = orders[orders['order_status'] == 'delivered']


In [5]:
merged = orders.merge(
    payments,
    on='order_id',
    how='left'
)


In [6]:
# Replace missing payments with 0
merged['payment_value'] = merged['payment_value'].fillna(0)


In [7]:
order_level = (
    merged
    .groupby(['order_id', 'customer_id', 'order_purchase_timestamp'])
    ['payment_value']
    .sum()
    .reset_index()
)

order_level.head()


,order_id,customer_id,order_purchase_timestamp,payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,72.19
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,259.83
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,216.87
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,218.04


In [ ]:
# Remove zero or negative payment orders
order_level = order_level[order_level['payment_value'] > 0]


In [9]:
Q1 = order_level['payment_value'].quantile(0.25)
Q3 = order_level['payment_value'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

order_level = order_level[
    (order_level['payment_value'] >= lower_bound) &
    (order_level['payment_value'] <= upper_bound)
]

print("After outlier removal:", order_level.shape)


After outlier removal: (88901, 4)


In [10]:
customer_df = (
    order_level
    .groupby('customer_id')
    .agg(
        total_spent=('payment_value', 'sum'),
        total_orders=('order_id', 'count'),
        last_purchase=('order_purchase_timestamp', 'max')
    )
    .reset_index()
)

customer_df.head()


,customer_id,total_spent,total_orders,last_purchase
0,00012a2ce6f8dcda20d059ce98491703,114.74,1,2017-11-14 16:08:26
1,000161a058600d5901f007fab4c27140,67.41,1,2017-07-16 09:40:32
2,0001fd6190edaaf884bcaf3d49edf079,195.42,1,2017-02-28 11:06:43
3,0002414f95344307404f0ace7a26f1d5,179.35,1,2017-08-16 13:09:20
4,000379cdec625522490c315e70c7a9fb,107.01,1,2018-04-02 13:42:17


In [11]:
# Remove customers with zero orders
customer_df = customer_df[customer_df['total_orders'] > 0]


In [12]:
latest_date = order_level['order_purchase_timestamp'].max()

customer_df['recency'] = (
    latest_date - customer_df['last_purchase']
).dt.days


In [13]:
customer_df['avg_order_value'] = (
    customer_df['total_spent'] / customer_df['total_orders']
)


In [14]:
BRL_TO_INR = 17.0

customer_df['total_spent'] = customer_df['total_spent'] * BRL_TO_INR
customer_df['avg_order_value'] = customer_df['avg_order_value'] * BRL_TO_INR


In [15]:
customer_df.isnull().sum()


customer_id        0
total_spent        0
total_orders       0
last_purchase      0
recency            0
avg_order_value    0
dtype: int64

In [17]:
customer_df.to_csv(
    "../data/preprocessed_customer_data.csv",
    index=False
)

customer_df.head()


,customer_id,total_spent,total_orders,last_purchase,recency,avg_order_value
0,00012a2ce6f8dcda20d059ce98491703,1950.58,1,2017-11-14 16:08:26,287,1950.58
1,000161a058600d5901f007fab4c27140,1145.97,1,2017-07-16 09:40:32,409,1145.97
2,0001fd6190edaaf884bcaf3d49edf079,3322.14,1,2017-02-28 11:06:43,547,3322.14
3,0002414f95344307404f0ace7a26f1d5,3048.95,1,2017-08-16 13:09:20,378,3048.95
4,000379cdec625522490c315e70c7a9fb,1819.17,1,2018-04-02 13:42:17,149,1819.17
